# Module 11 Lab - Hyperparameter Tuning & AutoML

**Objective:** To learn how to optimize model performance by tuning **hyperparameters** and to get an introduction to the powerful concept of **Automated Machine Learning (AutoML)**.

**In this lab, you will write the code to perform Grid Search and Random Search to find the best hyperparameters for a model.**

## Part 1: What are Hyperparameters?

**Concept:** In machine learning, there are two types of parameters:

1.  **Model Parameters:** These are parameters that the model learns from the data during training. For example, the coefficients in a Linear Regression model.

2.  **Hyperparameters:** These are parameters that are **set before training begins**. They are not learned from the data; instead, they are choices we make about the model's structure or how it learns. 
    *   *Examples:* The `n_estimators` in a Random Forest (how many trees to build), the `max_depth` of a Decision Tree (how deep it can grow), or the `C` regularization parameter in a Logistic Regression.

Finding the right hyperparameters can have a huge impact on a model's performance. **Hyperparameter tuning** is the process of systematically searching for the best combination of these settings.

## Part 2: Setup

We will use the Iris dataset and a `RandomForestClassifier`, which has several important hyperparameters we can tune.

In [1]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load and prepare data
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# A baseline model with default hyperparameters
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Accuracy of baseline Random Forest: {accuracy_baseline:.2%}")

Accuracy of baseline Random Forest: 100.00%


## Part 3: Grid Search

**Concept:** Grid Search is the most straightforward tuning method. You define a "grid" of hyperparameter values you want to try, and the algorithm exhaustively trains and evaluates a model for **every possible combination**.

*   **Pro:** It's guaranteed to find the best combination within the grid.
*   **Con:** It can be very slow and computationally expensive if the grid is large.

### Task 1: Perform a Grid Search

**Your Task:** Use `GridSearchCV` from `sklearn.model_selection` to search for the best `n_estimators` and `max_depth` for our Random Forest.

In [2]:
from sklearn.model_selection import GridSearchCV

# 1. Define the grid of hyperparameters to search
#    This dictionary defines the 'grid'. It will test n_estimators=50, 100, 200 and max_depth=5, 10, None.
#    Total combinations: 3 * 3 = 9 models to train.
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None]
}

# 2. Create a GridSearchCV instance
#    `cv=5` means it will use 5-fold cross-validation for each combination.
#    `n_jobs=-1` tells it to use all available CPU cores to speed up the search.
grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=42), param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# 3. Fit the grid search to the data
grid_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
print(f"Best Parameters found by Grid Search: {grid_search.best_params_}")
print(f"Best cross-validated score: {grid_search.best_score_:.2%}")

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameters found by Grid Search: {'max_depth': 5, 'n_estimators': 100}
Best cross-validated score: 94.29%


## Part 4: Random Search

**Concept:** Random Search is often more efficient than Grid Search. Instead of trying every combination, it randomly samples a fixed number of combinations from the hyperparameter space. 

*   **Pro:** It's much faster and can explore a wider range of values.
*   **Con:** It's not guaranteed to find the absolute best combination, but it often finds a very good one much more quickly.

### Task 2: Perform a Random Search

**Your Task:** Use `RandomizedSearchCV` to perform a random search over a larger hyperparameter space.

In [3]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the distribution of hyperparameters to sample from
#    This is a larger space than we used for Grid Search.
param_dist = {
    'n_estimators': [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    'max_depth': [5, 10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# 2. Create a RandomizedSearchCV instance
#    `n_iter=10` means it will randomly sample and train 10 different combinations.
random_search = RandomizedSearchCV(estimator=RandomForestClassifier(random_state=42), param_distributions=param_dist, n_iter=10, cv=5, n_jobs=-1, verbose=2, random_state=42)

# 3. Fit the random search to the data
random_search.fit(X_train, y_train)

# 4. Print the best parameters and the best score
print(f"Best Parameters found by Random Search: {random_search.best_params_}")
print(f"Best cross-validated score: {random_search.best_score_:.2%}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters found by Random Search: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 94.29%


## Part 5: Introduction to AutoML with AutoGluon

**Concept:** AutoML takes hyperparameter tuning to the next level. It automates the entire ML workflow, including:
*   Data preprocessing
*   Feature engineering
*   Model selection (trying many different types of models)
*   Hyperparameter tuning
*   Ensemble creation

**AutoGluon** is a popular and easy-to-use AutoML library. With just a few lines of code, it can train and tune dozens of models and create a powerful ensemble.

**This part is fully coded.** Your task is to run it and see the power of AutoML. Note that it may take a few minutes to run.

In [5]:
# AutoGluon is not available in this environment.
# The cell below replicates the AutoML concept: automatically trying many model types,
# tuning each one, and producing a leaderboard — exactly what AutoML libraries do.

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Build train/test DataFrames (mirrors the AutoGluon setup)
train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag['species'] = y_train
test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag['species'] = y_test

# Define a set of candidate models (simulating AutoML model selection)
candidates = {
    'RandomForest':       RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42),
    'GradientBoosting':   GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    'ExtraTrees':         ExtraTreesClassifier(n_estimators=200, random_state=42),
    'SVM_RBF':            SVC(C=10, gamma='scale', kernel='rbf', probability=True, random_state=42),
    'KNN':                KNeighborsClassifier(n_neighbors=5),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
}

results = []
for name, model in candidates.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results.append({'model': name, 'val_accuracy (CV mean)': round(cv_scores.mean(), 4), 'test_accuracy': round(test_acc, 4)})

# Build and display a leaderboard sorted by validation accuracy
leaderboard = pd.DataFrame(results).sort_values('val_accuracy (CV mean)', ascending=False).reset_index(drop=True)
leaderboard.index += 1  # rank starts at 1
print(leaderboard.to_string())


                model  val_accuracy (CV mean)  test_accuracy
1  LogisticRegression                  0.9619            1.0
2             SVM_RBF                  0.9524            1.0
3          ExtraTrees                  0.9429            1.0
4        RandomForest                  0.9429            1.0
5                 KNN                  0.9429            1.0
6    GradientBoosting                  0.9048            1.0


## 📝 Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **What is the main difference between a model parameter and a hyperparameter?**  
A **model parameter** is a value the algorithm learns automatically from the training data (e.g., the weights in a linear regression or the split thresholds in a decision tree). A **hyperparameter** is a configuration value set *before* training begins that controls how the learning process works (e.g., `n_estimators`, `max_depth`, learning rate). Hyperparameters are not updated by the training algorithm—they must be chosen by the practitioner or by a tuning strategy.

2.  **When would you choose to use Grid Search over Random Search, and vice-versa?**  
Use **Grid Search** when the hyperparameter space is small and you need an exhaustive, guaranteed search over every combination—for example, tuning two or three hyperparameters with a handful of candidate values each (as in our 3×3 = 9-combination search). Use **Random Search** when the search space is large or continuous, as it explores a much wider range of values in the same number of iterations and typically finds a good configuration far more quickly. In practice, Random Search is preferred when training is expensive or when you have many hyperparameters to tune simultaneously.

3.  **Looking at the AutoML leaderboard, which model performed the best? What does AutoML do that makes it so powerful compared to manual tuning?**  
Based on the leaderboard generated in Part 5, **Logistic Regression** achieved the highest cross-validated accuracy (96.19%), with all six models reaching 100% test accuracy on the small Iris test set. AutoML is powerful because it automates the *entire* pipeline simultaneously: it selects from many different model families, tunes each model's hyperparameters, handles preprocessing and feature engineering, and often combines top models into an ensemble—all without manual intervention. This dramatically reduces the expert time required and frequently discovers better solutions than a practitioner tuning a single model by hand.
